# Inteligencia de Negocios — Práctica 02
## Perfilamiento, limpieza y normalización con Python

**Propósito:** construir una versión de trabajo trazable del archivo de vivienda, aplicar reglas justificadas de limpieza y decidir si los registros resultantes están listos para un análisis posterior de precios por región y tipo de propiedad.

La práctica continúa el caso de la Práctica 01. Usaremos `Python` y `Pandas` con `datos/melbourne_housing-raw.csv`. El archivo crudo se conservará sin modificaciones.

## Problema organizacional y pregunta guía

Una organización interesada en estudiar la oferta de vivienda quiere comparar, en una etapa posterior, los precios registrados por región y tipo de propiedad. Antes de calcular métricas necesita saber si las fechas y categorías son comparables, cuánto dato falta y qué registros pueden entrar al análisis sin inventar información.

> **Pregunta guía:** ¿qué reglas de limpieza y qué verificaciones permiten preparar registros comparables sin ocultar faltantes ni alterar la fuente original?

La salida de esta práctica no es todavía un análisis de precios. Es una tabla de trabajo validada, una tabla analítica delimitada y una bitácora breve de decisiones que podrán reutilizarse en la transformación y el ETL.

## Fuente y metadatos indispensables

- **Conjunto de referencia:** [Melbourne Housing Market](https://www.kaggle.com/datasets/anthonypino/melbourne-housing-market), publicado en Kaggle por Tony Pino.
- **Procedencia declarada:** resultados inmobiliarios publicados por Domain.com.au y recopilados mediante *web scraping*.
- **Licencia declarada en Kaggle:** CC BY-NC-SA 4.0.
- **Archivo de trabajo:** copia del curso llamada `melbourne_housing-raw.csv`, con 21 columnas.
- **Unidad provisional:** registro de una propiedad asociado con ubicación, características, fecha y método de registro.
- **Columnas centrales para la pregunta:** `Price`, `Regionname`, `Type`, `Date` y `Method`.
- **Limitación documental:** el nombre del archivo local no conserva fecha de descarga ni número de versión; la trazabilidad de la copia es parcial.

Los códigos documentados que utilizaremos aparecen antes del Reto 3. No se deben inferir equivalencias adicionales.

## Ruta de trabajo

| Bloque | Producto mínimo | Tiempo orientativo |
|---|---|---:|
| Reto 1. Línea base | Copia de trabajo, columnas críticas y condiciones de calidad | 15 min |
| Reto 2. Perfil | Tabla de tipos, faltantes y valores distintos | 20 min |
| Reto 3. Normalización | Fecha y etiquetas documentadas sin sobrescribir el origen | 25 min |
| Reto 4. Reglas de limpieza | Tabla limpia y subconjunto analítico con conteos de impacto | 25 min |
| Reto 5. Validación | Comprobaciones, bitácora y decisión de preparación | 20 min |

**Ruta esencial:** los cinco retos, aproximadamente **105 minutos de trabajo en clase**.  
**Ampliación opcional:** revisar valores extremos de una columna adicional y documentar la regla que haría falta; 10–15 minutos. La ampliación no forma parte del producto mínimo.

## Cómo trabajar en este notebook

1. Ejecuta las celdas en orden y lee el ejemplo antes de cada reto.
2. Completa únicamente las celdas marcadas como **Código del reto** o **Respuesta del reto**.
3. No sobrescribas columnas crudas: crea columnas nuevas cuando normalices.
4. No rellenes ni elimines datos si la regla no está escrita en el notebook.
5. Si una comprobación técnica falla, conserva el mensaje y solicita apoyo; no instales paquetes durante la práctica.
6. Guarda una copia como `apellido_nombre_p02.ipynb`.

El código recupera evidencia. La calidad de la decisión se revisará por la relación entre **hallazgo, regla, operación, verificación e interpretación**, no sólo porque una celda se ejecute sin errores.

## Nivelación funcional: comprobar el entorno y cargar la fuente

La siguiente celda comprueba `Pandas`, localiza el CSV y crea una instantánea para verificar al final que el origen no cambió. `read_csv` interpreta el archivo como un `DataFrame`; no modifica el CSV.

In [4]:
from pathlib import Path
import pandas as pd

ruta = Path("datos") / "melbourne_housing-raw.csv"
assert ruta.exists(), f"No se encontró el archivo: {ruta.resolve()}"

datos_crudos = pd.read_csv(ruta)
datos_crudos_original = datos_crudos.copy(deep=True)

print(f"Pandas disponible: {pd.__version__}")
print(f"CSV disponible: {ruta.resolve()}")
print(f"Forma inicial: {datos_crudos.shape}")

Pandas disponible: 2.2.3
CSV disponible: /content/datos/melbourne_housing-raw.csv
Forma inicial: (19740, 21)


## Datos crudos, copia de trabajo y línea base

Una limpieza trazable separa el origen de la representación que puede cambiar. `copy(deep=True)` crea un `DataFrame` independiente; modificar la copia no altera el objeto original. La línea base registra qué recibimos antes de cualquier regla.

Para esta pregunta consideraremos **críticas** las columnas que permiten delimitar precio, región, tipo, fecha y método. Una condición de calidad debe expresar qué necesita el análisis: por ejemplo, una fecha interpretable o un código respaldado por el diccionario.

### Ejemplo resuelto: modificar una copia sin alterar el origen

In [7]:
origen_ejemplo = pd.DataFrame({
    "registro": [1, 2],
    "estado": ["pendiente", "confirmado"]
})
trabajo_ejemplo = origen_ejemplo.copy(deep=True)
trabajo_ejemplo.loc[0, "estado"] = "revisar"

print("Origen:")
print(origen_ejemplo)
print("\nCopia de trabajo:")
print(trabajo_ejemplo)
print("\n¿El origen se conservó?", origen_ejemplo.loc[0, "estado"] == "pendiente")

Origen:
   registro      estado
0         1   pendiente
1         2  confirmado

Copia de trabajo:
   registro      estado
0         1     revisar
1         2  confirmado

¿El origen se conservó? True


**Interpretación del ejemplo:** la separación permite experimentar con una representación nueva y contrastarla con lo recibido. No sustituye una bitácora, pero evita perder inmediatamente la evidencia original.

## Reto 1. Establecer la línea base y los criterios de calidad

Crea una copia de trabajo, registra la forma inicial y los duplicados exactos, y define las columnas críticas con una condición de calidad para cada una.

In [9]:
# Código del reto 1
# Plantilla reutilizable: completa las expresiones sin modificar datos_crudos.

datos_trabajo = datos_crudos.copy(deep=True)
linea_base = {
     "filas": datos_trabajo.shape[0],
     "columnas": datos_trabajo.shape[1],
     "duplicados_exactos": datos_trabajo.duplicated().sum(),
 }
print(linea_base)



{'filas': 19740, 'columnas': 21, 'duplicados_exactos': np.int64(1)}


### Respuesta del reto 1

**Pregunta analítica posterior:**
¿Cuál es la distribución y comparación de precios por región y tipo de propiedad en el mercado de vivienda de Melbourne?

**Forma inicial y duplicados exactos:**
19740 filas ,21 columnas y 1 duplicado exacto.

| Columna crítica | Por qué se necesita | Condición de calidad antes del análisis |
|---|---|---|
| `Price` | porque es la variable del objetivo numerico|Debe ser numérico noo nulo y mayor a cero |
| `Regionname` |Define el agrupamiento geográfico para comparar regiones | Debe contener una región válida y respaldada sin nulos.|
| `Type` | Clasifica la propiedad para la comparación por tipo|Debe corresponder a uno de los códigos documentados (h, u, t). |
| `Date` | Aporta la dimensión temporal para la comparabilidad de registros.| Debe poder ser convertida al formato de fecha día/mes/año (datetime)|
| `Method` |Identifica el estado/método de venta registrado. | Debe corresponder a una categoría válida según el diccionario.|


**Autocontrol 1:** antes de avanzar, confirma que `datos_trabajo is not datos_crudos` y que la forma de ambos objetos coincide. Contrasta tu línea base con otra persona o con la salida inicial.

## Perfil de calidad orientado a una decisión

Un perfil combina varias señales:

- `dtype` indica la representación técnica actual.
- `isna().sum()` cuenta ausencias, pero no explica su causa.
- `nunique(dropna=True)` cuenta valores distintos observados.
- `duplicated().sum()` detecta repeticiones exactas cuando no se especifica un subconjunto.

El porcentaje de faltantes facilita comparar columnas con el mismo número de filas. Una señal puede ser **bloqueante** para la pregunta, **relevante** pero tolerable, o quedar **pendiente de regla**.

### Ejemplo resuelto: perfil de tres columnas

In [10]:
columnas_ejemplo = ["Price", "Date", "Type"]
perfil_ejemplo = pd.DataFrame({
    "tipo_tecnico": datos_crudos[columnas_ejemplo].dtypes.astype(str),
    "faltantes": datos_crudos[columnas_ejemplo].isna().sum(),
    "porcentaje_faltante": (
        datos_crudos[columnas_ejemplo].isna().mean().mul(100).round(2)
    ),
    "valores_distintos": datos_crudos[columnas_ejemplo].nunique(dropna=True),
})
perfil_ejemplo

,tipo_tecnico,faltantes,porcentaje_faltante,valores_distintos
Price,float64,4344,22.01,2298
Date,object,0,0.00,52
Type,object,0,0.00,3


**Interpretación del ejemplo:** `Price` tiene ausencias que impiden usar algunas filas en una métrica de precio; `Date` y `Type` no tienen faltantes, pero su tipo `object` no demuestra que todas las fechas sean válidas ni que todos los códigos estén documentados. Perfilamiento no equivale a validación semántica.

## Reto 2. Construir e interpretar el perfil de calidad

Construye el perfil de las seis columnas indicadas, cuenta duplicados exactos y clasifica cada hallazgo como bloqueante, relevante o pendiente de regla.

In [12]:
# Código del reto 2
columnas_reto = [
    "Price", "Regionname", "Type", "Date", "Rooms", "BuildingArea"
]
perfil_reto = pd.DataFrame({
    "tipo_tecnico": datos_trabajo[columnas_reto].dtypes.astype(str),
    "faltantes": datos_trabajo[columnas_reto].isna().sum(),
    "porcentaje_faltante":datos_trabajo[columnas_reto].isna().mean().mul(100).round(2),
    "valores_distintos":datos_trabajo[columnas_reto].nunique(dropna=True),
    })
print(perfil_reto)
print("Duplicados exactos:", datos_trabajo.duplicated().sum())



             tipo_tecnico  faltantes  porcentaje_faltante  valores_distintos
Price             float64       4344                22.01               2298
Regionname         object          8                 0.04                  8
Type               object          0                 0.00                  3
Date               object          0                 0.00                 52
Rooms               int64          0                 0.00                 11
BuildingArea      float64      11123                56.35                915
Duplicados exactos: 1


### Respuesta del reto 2

| Hallazgo | Evidencia numérica | Clasificación | Consecuencia para la pregunta |
|---|---:|---|---|
| Faltantes en `Price` | 4,344|Bloqueante | Impide calcular métricas de precio directamente en esos registros|
| Faltantes en `Regionname` | 8| Bloqueante|Evita que esos registros puedan agruparse por región |
| `Date` como texto | 0|Relevante |Requiere conversión a datetime para realizar operaciones temporales |
| Códigos en `Type` |0 |Relevante | Requiere mapeo al nombre descriptivo según diccionario.|
| Faltantes en `BuildingArea` | 11,123| Pendiente de regla| No afecta la pregunta de precio/región actual, pero limita análisis físicos|
| Duplicados exactos |1 | Bloqueante|Debe eliminarse para no duplicar registros en el análisis |

**Hallazgo que debe atenderse antes de comparar precios:**
Los faltantes en Price y Regionname, así como el duplicado exacto, ya que distorsionan las comparaciones numéricas directas.

**Hallazgo que puede conservarse como limitación porque no interviene en la pregunta actual:**
La alta proporción de faltantes en BuildingArea (56.35%), dado que la métrica solicitada en esta etapa no exige metros cuadrados construidos.


**Autocontrol 2:** selecciona una cifra de tu perfil y recupérala con una operación independiente. Por ejemplo, contrasta el conteo de faltantes de una columna con `value_counts(dropna=False)`.

## Normalización documentada sin sobrescribir el origen

Normalizar significa llevar representaciones equivalentes a un formato común. En esta práctica crearemos columnas nuevas:

- `Date_clean`: fecha interpretada con formato día/mes/año.
- `Type_label`: descripción del tipo de propiedad.
- `Method_label`: descripción del método o resultado del registro.

`pd.to_datetime(..., errors="coerce")` convierte una entrada imposible en `NaT`; la hace visible como fallo de conversión, pero **no la corrige**. `map` produce un faltante cuando encuentra un código sin equivalencia, lo cual permite detectar valores no documentados.

### Diccionarios proporcionados

La documentación de la fuente describe `Type` como `h` (casa), `u` (unidad) y `t` (casa adosada). Para `Method`, los códigos observados se interpretan así: `S` (vendida), `SP` (vendida previamente), `PI` (no adjudicada en subasta), `PN` (vendida previamente, precio no divulgado), `SN` (vendida, precio no divulgado), `VB` (oferta del vendedor), `W` (retirada), `SA` (vendida después de subasta) y `SS` (vendida después de subasta, precio no divulgado).

In [13]:
mapa_tipo = {
    "h": "casa",
    "u": "unidad",
    "t": "casa adosada",
}

mapa_metodo = {
    "S": "vendida",
    "SP": "vendida previamente",
    "PI": "no adjudicada en subasta",
    "PN": "vendida previamente, precio no divulgado",
    "SN": "vendida, precio no divulgado",
    "VB": "oferta del vendedor",
    "W": "retirada",
    "SA": "vendida después de subasta",
    "SS": "vendida después de subasta, precio no divulgado",
}

### Ejemplo resuelto: convertir y comprobar una muestra

In [14]:
muestra_codigos = pd.DataFrame({
    "Date": ["3/09/2016", "31/02/2017"],
    "Type": ["h", "x"],
    "Method": ["S", "ZZ"],
})
muestra_normalizada = muestra_codigos.copy()
muestra_normalizada["Date_clean"] = pd.to_datetime(
    muestra_normalizada["Date"], format="%d/%m/%Y", errors="coerce"
)
muestra_normalizada["Type_label"] = muestra_normalizada["Type"].map(mapa_tipo)
muestra_normalizada["Method_label"] = muestra_normalizada["Method"].map(mapa_metodo)

print(muestra_normalizada)
print("Fechas no convertibles:", muestra_normalizada["Date_clean"].isna().sum())
print("Tipos sin equivalencia:", muestra_normalizada["Type_label"].isna().sum())

         Date Type Method Date_clean Type_label Method_label
0   3/09/2016    h      S 2016-09-03       casa      vendida
1  31/02/2017    x     ZZ        NaT        NaN          NaN
Fechas no convertibles: 1
Tipos sin equivalencia: 1


**Interpretación del ejemplo:** la primera fila queda normalizada. La segunda conserva los valores crudos, mientras `NaT` y los faltantes de las etiquetas señalan que se requiere revisión. Ocultar esos fallos con una etiqueta inventada eliminaría evidencia.

## Reto 3. Normalizar fecha y categorías documentadas

Crea columnas normalizadas para fecha, tipo y método; comprueba los fallos de conversión y los códigos no documentados sin sobrescribir las columnas de origen.

In [16]:
# Código del reto 3
# Adapta el ejemplo al DataFrame datos_trabajo.
datos_trabajo["Date_clean"] = pd.to_datetime(
     datos_trabajo["Date"], format="%d/%m/%Y", errors="coerce"
 )
datos_trabajo["Type_label"] = datos_trabajo["Type"].map(mapa_tipo)
datos_trabajo["Method_label"] = datos_trabajo["Method"].map(mapa_metodo)
print("Fechas no convertibles:", datos_trabajo["Date_clean"].isna().sum())
print("Tipos no documentados:", datos_trabajo["Type_label"].isna().sum())
print("Métodos no documentados:", datos_trabajo["Method_label"].isna().sum())
datos_trabajo[["Date", "Date_clean", "Type", "Type_label", "Method", "Method_label"]].head()


Fechas no convertibles: 0
Tipos no documentados: 0
Métodos no documentados: 0


,Date,Date_clean,Type,Type_label,Method,Method_label
0,3/09/2016,2016-09-03,h,casa,SS,"vendida después de subasta, precio no divulgado"
1,3/12/2016,2016-12-03,h,casa,S,vendida
2,4/02/2016,2016-02-04,h,casa,S,vendida
3,4/02/2016,2016-02-04,u,unidad,VB,oferta del vendedor
4,4/03/2017,2017-03-04,h,casa,SP,vendida previamente


### Respuesta del reto 3

**Fechas no convertibles:**
0

**Códigos de tipo no documentados:**
0

**Códigos de método no documentados:**
0

**¿Qué evidencia muestra que las columnas crudas se conservaron?**
Que mantuve intactas las columnas originales (Date, Type, Method) y cree columnas nuevas con otros nombres (Date_clean, Type_label, Method_label) para hacer las conversiones. Puedo comprobarlo haciendo un .head() de las columnas viejas junto a las nuevas.

**¿Qué decisión de comparabilidad permite el resultado?**
Garantiza que todas las fechas ya se entienden como fechas reales y que todos los tipos y métodos están bien traducidos según el diccionario, sin valores raros o inventados.

**Autocontrol 3:** muestra lado a lado `Date` y `Date_clean`, así como cada código y su etiqueta. Si aparece un faltante nuevo, vuelve al valor crudo y verifica si se trata de una ausencia original o de una equivalencia no documentada.

## Reglas explícitas para duplicados y faltantes

Aplicaremos sólo estas reglas:

1. Una fila idéntica en las 21 columnas crudas representa una repetición técnica para esta copia y se conserva una sola vez en `datos_limpios`.
2. `Price` no se imputa: no existe una regla documentada para reconstruirlo.
3. La tabla `datos_limpios` conserva faltantes que no están autorizados para modificarse.
4. Para la pregunta posterior se crea `analisis_precios`, que incluye únicamente filas con `Price`, `Regionname`, `Type_label` y `Date_clean` presentes. Esta exclusión delimita el análisis; no borra los registros de la tabla limpia.

Separar una tabla limpia de un subconjunto analítico hace visible qué registros quedan fuera de una pregunta concreta.

### Ejemplo resuelto: una regla distinta para cada señal

In [18]:
muestra_limpieza = pd.DataFrame({
    "registro": [1, 1, 2, 3],
    "precio": [100.0, 100.0, None, 140.0],
    "region": ["Norte", "Norte", "Sur", "Sur"],
})
muestra_sin_duplicados = muestra_limpieza.drop_duplicates().copy()
muestra_para_precio = muestra_sin_duplicados.dropna(
    subset=["precio", "region"]
).copy()

print("Filas recibidas:", len(muestra_limpieza))
print("Filas únicas:", len(muestra_sin_duplicados))
print("Filas aptas para analizar precio:", len(muestra_para_precio))

Filas recibidas: 4
Filas únicas: 3
Filas aptas para analizar precio: 2


**Interpretación del ejemplo:** una fila se elimina por igualdad completa y otra queda fuera del análisis de precio porque no tiene esa medida. El valor ausente permanece visible en la tabla sin duplicados y no se convierte en cero ni en un promedio inventado.

## Reto 4. Aplicar reglas de limpieza y medir su impacto

Elimina sólo duplicados exactos en la tabla limpia, crea el subconjunto con las cuatro columnas indispensables y registra cuántas filas afecta cada regla sin imputar valores.

In [23]:
# Código del reto 4
datos_limpios = datos_trabajo.drop_duplicates(
     subset=datos_crudos.columns
 ).copy()
columnas_indispensables = [
     "Price", "Regionname", "Type_label", "Date_clean"
 ]

analisis_precios = datos_limpios.dropna(
     subset=columnas_indispensables
).copy()
impacto = pd.Series({
    "filas_recibidas": len(datos_trabajo),
    "duplicados_retirados": len(datos_trabajo),
    "filas_en_tabla_limpia": len(datos_limpios),
    "filas_aptas_para_precios": len(analisis_precios),
    "filas_fuera_del_analisis": len(datos_limpios)-len(analisis_precios),
})
print(impacto)
print("Precios nulos en datos_limpios:", datos_limpios["Price"].isna().sum())
print("Precios nulos en analisis_precios:", analisis_precios["Price"].isna().sum())
print("¿La suma de filas cuadra?:", len(analisis_precios) + (len(datos_limpios) - len(analisis_precios)) == len(datos_limpios))
print("Porcentaje de datos aptos:", round((len(analisis_precios) / len(datos_trabajo)) * 100, 2), "%")


filas_recibidas             19740
duplicados_retirados        19740
filas_en_tabla_limpia       19739
filas_aptas_para_precios    15389
filas_fuera_del_analisis     4350
dtype: int64
Precios nulos en datos_limpios: 4343
Precios nulos en analisis_precios: 0
¿La suma de filas cuadra?: True
Porcentaje de datos aptos: 77.96 %


### Respuesta del reto 4

| Regla | Filas antes | Filas después | Evidencia conservada |
|---|---:|---:|---|
| Retirar duplicados exactos | 19740|19739 | Se borró únicamente 1 fila totalmente repetida en todas sus columnas.|
| Delimitar registros para analizar precios |19,739 |15,389 |datos_limpios guarda todas las filas; analisis_precios solo toma las que tienen los 4 datos obligatorios. |

**Porcentaje de registros recibidos que entra al análisis de precios:**
77.96%

**Por qué no se imputó `Price`:**

Porque no tenemos una regla documentada ni justificación de negocio para inventar el precio de una casa sin alterar los resultados reales.
**Limitación introducida por excluir filas del análisis:**
Que nuestro análisis de precios solo aplica para las casas que sí hicieron público su precio, dejando fuera el 22% de las propiedades registradas.


**Autocontrol 4:** comprueba que `datos_limpios` conserva las filas sin precio y que sólo `analisis_precios` las excluye. La suma de filas aptas y filas fuera del análisis debe coincidir con el tamaño de la tabla limpia.

## Validación y decisión de preparación

Una validación convierte las reglas en condiciones comprobables. Una aserción que falla detiene la ejecución y señala que el resultado no cumple lo esperado; no demuestra por sí sola que todos los datos sean correctos.

Para cerrar necesitamos verificar: conservación del origen, ausencia de duplicados exactos en la tabla limpia, conversiones documentadas y presencia de las cuatro columnas indispensables en el subconjunto analítico. Después interpretaremos el impacto y elegiremos una decisión: **lista**, **lista con cautela** o **pendiente de revisión**.

### Ejemplo resuelto: convertir reglas en comprobaciones

In [25]:
comprobaciones_ejemplo = pd.Series({
    "sin_duplicados_exactos": muestra_sin_duplicados.duplicated().sum() == 0,
    "subconjunto_sin_precio_faltante": muestra_para_precio["precio"].notna().all(),
    "origen_conservado": len(muestra_limpieza) == 4,
})
print(comprobaciones_ejemplo)
assert comprobaciones_ejemplo.all(), "Alguna regla del ejemplo no se cumplió"

sin_duplicados_exactos             True
subconjunto_sin_precio_faltante    True
origen_conservado                  True
dtype: bool


**Interpretación del ejemplo:** las tres condiciones verdaderas autorizan usar el subconjunto para la pregunta definida. No autorizan a generalizar los resultados a registros sin precio ni a afirmar que los valores extremos son correctos.

## Reto 5. Validar el resultado y decidir si está preparado

Construye y ejecuta cinco comprobaciones, completa la bitácora y decide si el resultado está listo, listo con cautela o pendiente para el análisis posterior de precios.

In [27]:
# Código del reto 5
comprobaciones = pd.Series({
   "origen_conservado": datos_crudos.equals(datos_crudos_original),
   "sin_duplicados_exactos": datos_crudos.equals(datos_crudos_original),
   "fechas_convertidas": datos_limpios.duplicated().sum() == 0,
   "tipos_y_metodos_documentados": (datos_trabajo["Type_label"].notna().all()) and (datos_trabajo["Method_label"].notna().all()),
   "subconjunto_sin_nulos_indispensables": analisis_precios[columnas_indispensables].notna().all().all(),
})
print(comprobaciones)
assert comprobaciones.all(), "Revisa las reglas que no se cumplieron"



origen_conservado                       True
sin_duplicados_exactos                  True
fechas_convertidas                      True
tipos_y_metodos_documentados            True
subconjunto_sin_nulos_indispensables    True
dtype: bool


### Respuesta del reto 5

| Hallazgo inicial | Regla aplicada | Verificación | Resultado o limitación |
|---|---|---|---|
| Fila exactamente repetida |Desduplicar la tabla | duplicated().sum() == 0| Se eliminó el duplicado exacto. |
| Fechas almacenadas como texto |  Convertir a datetime |Date_clean.notna().all() |El 100% de las fechas se convirtió correctamente|
| Códigos de tipo y método | Traducir según diccionario| 	Type_label y Method_label sin nulos	 | Todas las categorías quedaron mapeadas|
| Precios faltantes | Filtrar subconjunto analítico|Price.notna().all() en subconjunto | Se excluyeron 4,350 filas sin precio del análisis|
| Otras columnas con faltantes |Conservar nulos en tabla limpia| Mantenidos sin modificar|Se conservan faltantes en variables como BuildingArea.  |

**Decisión de preparación:**
Lista con cautela

**Evidencia que sustenta la decisión:**
Todas las comprobaciones pasaron en True, el archivo original no sufrió modificaciones y generamos un subconjunto analítico de 15,389 filas válidas y sin faltantes en sus variables indispensables.


**Qué decisión organizacional sí podría apoyarse después:**
Comparar los precios promedios y medianos por región y tipo de propiedad para los casos con precio registrado.


**Qué afirmación todavía no está justificada:**

Afirmar que estos precios representan a todo el mercado de Melbourne, pues dejamos fuera un 22% de propiedades que no publicaron su costo.

**Autocontrol 5:** ninguna comprobación debe quedar en `False`. Si todas son verdaderas, revisa además que la decisión escrita mencione los registros excluidos y la trazabilidad parcial de la copia local; ejecutar sin errores no elimina esas limitaciones.

## Ampliación opcional

Elige `Bedroom2`, `BuildingArea`, `Landsize` o `YearBuilt` y localiza valores ausentes o extremos. Registra la señal y explica qué regla de dominio o metadato necesitarías antes de modificarla. **No apliques una corrección automática.**

| Columna | Señal observada | Evidencia | Regla o metadato necesario |
|---|---|---|---|
|BuildingArea | 56.35 % de valores ausentes y registros con 0 $m^2$|isna().sum() == 11123 y valores mínimos en 0| Confirmar con la fuente de donde se extrajeron los datos, si el 0 representa un faltante no registrado o un terreno sin edificación|


## Evidencias de entrega y criterios de revisión

- [x] Conservé el CSV y `datos_crudos` sin modificaciones.
- [x] Ejecuté los ejemplos y completé el código de los cinco retos.
- [x] Relacioné el perfil con la pregunta de negocio.
- [x] Apliqué únicamente las reglas proporcionadas y medí su impacto.
- [x] Conservé valores crudos junto a sus representaciones normalizadas.
- [x] Todas las comprobaciones finales son verdaderas o documenté el fallo.
- [x] Justifiqué la decisión de preparación y sus limitaciones.

La revisión considerará cinco aspectos: **trazabilidad del origen, perfil correcto, aplicación justificada de reglas, verificaciones reproducibles e interpretación para la decisión de negocio**.